# TinyLoRA: a handful of trainable numbers

TinyLoRA (from the paper *Learning to Reason in 13 Parameters*) pushes LoRA to the extreme. It builds on LoRA-XS and has three ideas:

1. **Use the weight's own directions.** A truncated SVD gives `W ≈ U Σ Vᵀ` with rank `r`. These directions are frozen. Only a tiny `r × r` matrix `R` in between is learned:
   $$\Delta W = U \Sigma\, R\, V^\top$$
2. **Project `R` from a short vector.** `R` itself is not trained. It is a mix of `N` fixed random matrices `P_i`, with weights from a trainable vector `v`:
   $$R = \sum_{i=1}^{N} v_i P_i \quad\Rightarrow\quad \Delta W = \sum_{i=1}^{N} v_i\, \underbrace{U \Sigma P_i V^\top}_{\lambda_i \text{ (fixed)}}$$
   This is the picture: `ΔW = v · λ`, where `λ` is a fixed `N × d_out × d_in` tensor.
3. **Tie `v` across layers.** Every adapted matrix in the model shares the **same** `v`. With `N = 13`, the whole model has 13 trainable numbers.

`v` starts at zero, so the model starts out as the base model.

**Caveat.** The 13-parameter result in the paper comes from **reinforcement learning** on math reasoning, where the model mostly needs to be nudged toward behavior it already has. This notebook uses plain SFT on a new output format, which needs more information than 13 numbers can hold. Expect the loss to drop, but less than with LoRA. `TIE` below lets you untie `v` per block or per layer to see how the result grows with the parameter count.

## The picture in numbers

The toy matrix: `W` is 800 × 500, rank `r = 4`, `N = 13`. Building `λ` explicitly and contracting it with `v` gives the same `ΔW` as the cheap form `U Σ R Vᵀ`.

In [ ]:
import torch

torch.manual_seed(0)
d_out, d_in, r, n = 800, 500, 4, 13
W = torch.randn(d_out, d_in)
U, S, Vh = torch.linalg.svd(W, full_matrices=False)
US, Vh_r = U[:, :r] * S[:r], Vh[:r]
P = torch.randn(n, r, r) / r**0.5

lam = torch.einsum("or,nrs,si->noi", US, P, Vh_r)  # fixed tensor λ: N x 800 x 500
v = torch.randn(n) * 0.01                          # the only trainable numbers

delta_from_lambda = torch.einsum("n,noi->oi", v, lam)
delta_cheap = US @ torch.einsum("n,nrs->rs", v, P) @ Vh_r

print("λ shape:", tuple(lam.shape))
print("v · λ == U Σ R Vᵀ:", torch.allclose(delta_from_lambda, delta_cheap, atol=1e-4))
print(f"full finetuning: {W.numel():,} parameters")
print(f"LoRA r=4:        {d_out * r + r * d_in:,} parameters")
print(f"TinyLoRA:        {v.numel()} parameters")

## Setup

Same base model and tokenizer as `finetuning.ipynb`. Run `shop_assistant_dataset.ipynb` first so `data/instruction-data.json` exists.

In [ ]:
import json
import sys
import time
from functools import partial
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from reasoning_from_scratch.qwen3 import (
    QWEN_CONFIG_06_B,
    KVCache,
    Qwen3Model,
    Qwen3Tokenizer,
    download_qwen3_small,
)

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("No CUDA GPU found. Everything runs on CPU, but training will be slow.")

download_qwen3_small(kind="base", out_dir=ROOT / "qwen3")
tokenizer = Qwen3Tokenizer(ROOT / "qwen3" / "tokenizer-base.json")
model = Qwen3Model(QWEN_CONFIG_06_B)
weights = torch.load(ROOT / "qwen3" / "qwen3-0.6B-base.pth", map_location="cpu", weights_only=True)
model.load_state_dict(weights)
del weights
model.to(device)
model.eval()

n_base = sum(param.numel() for param in model.parameters())
print(f"base model: {n_base / 1e6:.1f}M parameters on {device}")

In [ ]:
INSTRUCTION = (
    "You are a shop assistant for a clothing and shoe store. "
    "Read the customer message and return only the next action as JSON "
    "with keys action, slots, missing, and reply. "
    "Do not invent products, prices, or stock."
)


def format_input(entry: dict) -> str:
    """Format an instruction entry as the training prompt."""
    instruction_text = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text


class InstructionDataset(Dataset):
    """Prompt plus response, with the prompt length kept for the loss mask."""

    def __init__(self, data: list[dict], tokenizer) -> None:
        self.examples = []
        for entry in data:
            prompt = format_input(entry) + "\n\n### Response:\n"
            full = prompt + entry["output"]
            prompt_ids = tokenizer.encode(prompt)
            full_ids = tokenizer.encode(full)
            self.examples.append((full_ids, len(prompt_ids)))

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.examples[index]

    def __len__(self) -> int:
        return len(self.examples)


def collate(batch, device, pad_id: int):
    """Pad a batch and ignore prompt and padding tokens in the loss."""
    sequences, prompt_lens = [], []
    for ids, prompt_len in batch:
        sequences.append(ids + [pad_id])
        prompt_lens.append(prompt_len)
    max_len = max(len(seq) for seq in sequences)
    input_rows, target_rows = [], []
    for seq, prompt_len in zip(sequences, prompt_lens):
        padded = seq + [pad_id] * (max_len - len(seq))
        inputs = torch.tensor(padded[:-1], device=device)
        targets = torch.tensor(padded[1:], device=device)
        targets[: prompt_len - 1] = -100
        pad_positions = (targets == pad_id).nonzero(as_tuple=True)[0]
        if len(pad_positions) > 1:
            targets[pad_positions[1:]] = -100
        input_rows.append(inputs)
        target_rows.append(targets)
    return torch.stack(input_rows), torch.stack(target_rows)


with open(ROOT / "data" / "instruction-data.json", encoding="utf-8") as f:
    data = json.load(f)

train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
train_data = data[:train_portion]
test_data = data[train_portion : train_portion + test_portion]
val_data = data[train_portion + test_portion :]

collate_fn = partial(collate, device=device, pad_id=tokenizer.pad_token_id)
train_loader = DataLoader(
    InstructionDataset(train_data, tokenizer),
    batch_size=1,
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
val_loader = DataLoader(
    InstructionDataset(val_data, tokenizer),
    batch_size=1,
    collate_fn=collate_fn,
    shuffle=False,
    num_workers=0,
)
print("train", len(train_data), "val", len(val_data), "test", len(test_data))

In [ ]:
# Every linear layer inside the 28 transformer blocks gets an adapter.
# The token embedding, the norms, and the output head stay as they are.
TARGETS = {"att": ("W_query", "W_key", "W_value", "out_proj"), "ff": ("fc1", "fc2", "fc3")}


def target_layers(model):
    """Yield (parent module, attribute name) for every adapted linear layer."""
    for block in model.trf_blocks:
        for parent_name, names in TARGETS.items():
            parent = getattr(block, parent_name)
            for name in names:
                yield parent, name


def add_adapters(model, wrap) -> None:
    """Freeze every base weight, then replace each target layer with wrap(layer)."""
    for param in model.parameters():
        param.requires_grad = False
    for parent, name in target_layers(model):
        setattr(parent, name, wrap(getattr(parent, name)))


def count_trainable(model) -> int:
    """Count trainable parameters. Shared parameters are counted once."""
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


@torch.no_grad()
def evaluate_loss(model, loader) -> float:
    """Mean masked cross-entropy over a loader."""
    model.eval()
    losses = []
    for inputs, targets in loader:
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
        losses.append(loss.item())
    return sum(losses) / len(losses)


def train(model, params, lr: float, accum_steps: int = 4, weight_decay: float = 0.0, log_every: int = 40) -> None:
    """One pass over the training split. Gradients from 4 examples make one AdamW step."""
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    optimizer.zero_grad()
    start = time.time()
    for step, (inputs, targets) in enumerate(train_loader, start=1):
        logits = model(inputs)
        loss = nn.functional.cross_entropy(logits.flatten(0, 1).float(), targets.flatten())
        (loss / accum_steps).backward()
        if step % accum_steps == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()
        if step % log_every == 0 or step == len(train_loader):
            print(f"step {step}/{len(train_loader)} loss {loss.item():.3f}")
    model.eval()
    print(f"training took {time.time() - start:.0f}s")
    if device.type == "cuda":
        print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")


def prompt_for(customer: str) -> str:
    """Wrap a customer message in the training prompt."""
    return format_input({"instruction": INSTRUCTION, "input": customer}) + "\n\n### Response:\n"


def generate(model, prompt: str, max_new_tokens: int = 100) -> str:
    """Greedy decoding with a KV cache. Stops at <|endoftext|>."""
    token_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    new_ids = []
    with torch.inference_mode():
        logits = model(token_ids, cache=cache)
        for _ in range(max_new_tokens):
            next_id = logits[:, -1].argmax(dim=-1, keepdim=True)
            if next_id.item() == tokenizer.eos_token_id:
                break
            new_ids.append(next_id.item())
            logits = model(next_id, cache=cache)
    model.reset_kv_cache()
    return tokenizer.decode(new_ids).strip()


def score_actions(model, rows: list[dict]) -> None:
    """Generate an action for each row and compare it with the expected one."""
    valid = correct = 0
    for entry in rows:
        text = generate(model, prompt_for(entry["input"]))
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            continue
        if not isinstance(payload, dict):
            continue
        valid += 1
        correct += payload.get("action") == json.loads(entry["output"])["action"]
    print(f"valid JSON {valid}/{len(rows)}   right action {correct}/{len(rows)}")


SAMPLE = "Busco tenis para correr, talla 42, que no sean negros"

## The TinyLoRA layer

Each linear layer stores its own frozen `U Σ` (d_out × r), `Vᵀ` (r × d_in) and random `P` (N × r × r). The trainable `v` is passed in, so many layers can hold the same `Parameter`. `model.parameters()` lists a shared parameter only once, which makes the count below exact.

`TIE` chooses how much is shared:

| `TIE` | one `v` per | trainable numbers (N = 13) |
|---|---|---|
| `"all"` | whole model | 13 |
| `"block"` | transformer block | 13 × 28 = 364 |
| `"none"` | linear layer | 13 × 196 = 2,548 |

The SVD of 196 matrices takes a few seconds on the GPU.

In [ ]:
class TinyLoRALinear(nn.Module):
    """Frozen linear layer plus U Σ (Σ_i v_i P_i) Vᵀ with a shared trainable vector v."""

    def __init__(self, base: nn.Linear, v: nn.Parameter, rank: int, seed: int) -> None:
        super().__init__()
        self.base = base
        self.v = v
        U, S, Vh = torch.linalg.svd(base.weight.detach().float(), full_matrices=False)
        self.register_buffer("US", U[:, :rank] * S[:rank])
        self.register_buffer("Vh", Vh[:rank].contiguous())
        generator = torch.Generator().manual_seed(seed)
        P = torch.randn(len(v), rank, rank, generator=generator) / rank**0.5
        self.register_buffer("P", P.to(base.weight.device))

    def small_matrix(self) -> torch.Tensor:
        """R = Σ_i v_i P_i, an r x r matrix."""
        return torch.einsum("n,nij->ij", self.v, self.P)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        update = ((x.float() @ self.Vh.T) @ self.small_matrix().T) @ self.US.T
        return self.base(x) + update.to(x.dtype)

    def delta_weight(self) -> torch.Tensor:
        """The full d_out x d_in update this adapter represents."""
        return self.US @ self.small_matrix() @ self.Vh


RANK = 4
N_PARAMS = 13
TIE = "all"  # "all", "block", or "none"

shared_vectors: dict[int, nn.Parameter] = {}
tiny_layers: list[TinyLoRALinear] = []
layers_per_block = sum(len(names) for names in TARGETS.values())


def wrap(layer: nn.Linear) -> TinyLoRALinear:
    """Wrap one layer and pick which v it shares."""
    index = len(tiny_layers)
    key = {"all": 0, "block": index // layers_per_block, "none": index}[TIE]
    if key not in shared_vectors:
        shared_vectors[key] = nn.Parameter(torch.zeros(N_PARAMS, device=device))
    tiny = TinyLoRALinear(layer, shared_vectors[key], RANK, seed=index)
    tiny_layers.append(tiny)
    return tiny


add_adapters(model, wrap)

n_trainable = count_trainable(model)
print(f"{len(tiny_layers)} adapted layers sharing {len(shared_vectors)} vector(s)")
print(f"trainable {n_trainable:,} / {n_base:,} base parameters ({100 * n_trainable / n_base:.7f}%)")

In [ ]:
val_before = evaluate_loss(model, val_loader)
print(f"validation loss before training: {val_before:.3f}")
print("base model output:")
print(generate(model, prompt_for(SAMPLE)))

## Training

Same loop as the other notebooks. With so few parameters the learning rate is much higher than LoRA's 2e-4: every step of `v` moves 196 matrices at once. Treat it as a knob.

In [ ]:
tiny_params = list(shared_vectors.values())
train(model, tiny_params, lr=1e-3)

In [ ]:
val_after = evaluate_loss(model, val_loader)
print(f"validation loss: {val_before:.3f} -> {val_after:.3f}")
print("finetuned output:")
print(generate(model, prompt_for(SAMPLE)))
score_actions(model, test_data)

## The whole adapter

With `TIE = "all"` the trained adapter is 13 numbers. The fixed parts (`U Σ`, `Vᵀ`, `P`) can be rebuilt from the base weights and the seeds, so these numbers and the settings are all you need to store. Merging works the same as in LoRA.

In [ ]:
adapter = {
    "rank": RANK,
    "n_params": N_PARAMS,
    "tie": TIE,
    "vectors": {key: vector.detach().cpu() for key, vector in shared_vectors.items()},
}
adapter_path = ROOT / "qwen3" / "shop-tinylora-adapter.pth"
torch.save(adapter, adapter_path)
print(f"adapter file: {adapter_path.stat().st_size:,} bytes")
if TIE == "all":
    print("the trained numbers:", [round(value, 4) for value in shared_vectors[0].tolist()])


@torch.no_grad()
def merge_adapters(model) -> None:
    """Add each adapter's update into its base weight and drop the adapter."""
    for parent, name in target_layers(model):
        layer = getattr(parent, name)
        layer.base.weight += layer.delta_weight().to(layer.base.weight.dtype)
        setattr(parent, name, layer.base)


merge_adapters(model)
print(generate(model, prompt_for(SAMPLE)))

## Recap

- `ΔW` lives in each matrix's top-`r` singular directions, and inside that space it can only move along `N` fixed random directions. Tying shares those `N` coordinates across the whole model.
- This is the far end of the trade-off. It is great for storage and for steering a capable model with RL. It is weak for teaching a new skill or format with SFT.
- Compare with the other notebooks: 218.5M (partial finetune) → ~5M (LoRA) → ~2.8M (LoRA-FA) → 13 (TinyLoRA).